In [1]:
import sys

print("Python version: ", sys.version)
print("Python executable: ", sys.executable)

Python version:  3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Python executable:  /Users/timmayabi/Projects/newsfindr/.venv/bin/python


In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

#load environment variables from .env file
load_dotenv()

#verify that the Groq API exists
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Groq API key not found in environment variables.")
else:
    print("Groq API key found.")


Groq API key found.



### Observation

The Groq API key was successfully retrieved from the
local environment configuration using python-dotenv.

The API key is stored separately from the notebook
to prevent accidental exposure of sensitive credentials.

### Initializing the Groq LLM

In [3]:
# Model configuration
MODEL_NAME = "openai/gpt-oss-120b"

# Intitialize Groq LLM
llm = ChatGroq(
    model_name=MODEL_NAME,
    temperature=0,
    max_tokens=1000,
    max_retries=2,
    groq_api_key=groq_api_key
)
print("Groq LLM initialized successfully.")
print("Model:", MODEL_NAME)

Groq LLM initialized successfully.
Model: openai/gpt-oss-120b


In [5]:
# Simple query to verify Groq API connectivity
query = "What is Agentic AI? Explain in two sentences."
response = llm.invoke(query)
print("Query: ", query)
print("\nLLM response:")
print(response.content)

Query:  What is Agentic AI? Explain in two sentences.

LLM response:
Agentic AI refers to artificial intelligence systems designed to act autonomously toward specific goals, making decisions, planning, and executing actions without constant human oversight. These agents combine perception, reasoning, and learning capabilities to interact with their environment and adapt their behavior over time.


In [6]:
# Inspect token usage reported by Groq

usage = response.usage_metadata or {}

print("Input tokens:", usage.get("input_tokens", "N/A"))
print("Output tokens:", usage.get("output_tokens", "N/A"))
print("Total tokens:", usage.get("total_tokens", "N/A"))

Input tokens: 82
Output tokens: 91
Total tokens: 173



### Observations

- The Groq-hosted LLM was initialized using LangChain.
- The model successfully processed a simple natural-language query.
- A valid response was returned, confirming API connectivity.
- Token usage information was retrieved to support future
  rate-limit monitoring.

### Conclusion

The LLM is operational and ready for integration with
the SQL and news retrieval agents.


In [9]:
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, inspect

# Locate the customer database

DB_PATH = Path.cwd() / "customer.db"

#Check whether the database file exists
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database file not found at {DB_PATH}")
else:
    print(f"Database file found at {DB_PATH}")
    print("Database size:", DB_PATH.stat().st_size, "bytes")

Database file found at /Users/timmayabi/Projects/newsfindr/customer.db
Database size: 20480 bytes


In [10]:
# Connect to the customer database using SQLAlchemy
engine = create_engine(f"sqlite:///{DB_PATH}")
inspector = inspect(engine)
print("Tables in the database:", inspector.get_table_names())

# Optionally, inspect the columns of each table
for table_name in inspector.get_table_names():
    print(f"\nColumns in table '{table_name}':")
    for column in inspector.get_columns(table_name):
        print(f" - {column['name']} ({column['type']})")

Tables in the database: ['customers']

Columns in table 'customers':
 - id (INTEGER)
 - customer_id (TEXT)
 - name (TEXT)
 - email (TEXT)
 - interests (TEXT)
 - last_updated (TIMESTAMP)


In [14]:
# Verify the number of customer records
from sqlalchemy import text
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM customers"))
    customer_count = result.scalar()
print("Number of customer records:", customer_count)

Number of customer records: 15
